# The Optimization Landscape

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.01–02.10, 03.01–03.11 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/01_optimization_landscape.ipynb)

---

## 🎯 Learning Objective

Map the vocabulary of optimization landscapes — local/global minima, saddle points, critical points — onto the loss surfaces neural networks actually train on.

---

## 📐 Theory

Every optimization algorithm in this module explores a **loss landscape** — the graph of
$L(\mathbf{w})$ over parameter space. Before studying *how* to navigate it (`04.02` onward), we
need vocabulary for its terrain, and an answer to the question motivating this module: loss
surfaces are wildly non-convex, so why does gradient descent reliably find good solutions?

### Critical points

A point $\mathbf{w}^*$ is **critical** if $\nabla L(\mathbf{w}^*)=\mathbf{0}$. Recall from
`02.05` that the gradient points toward steepest ascent, so a zero gradient means no local
direction increases $L$ — gradient descent can only stop at a critical point. The Hessian
$H=\nabla^2 L(\mathbf{w}^*)$ from `02.06` classifies each one via its eigenvalues: **local
minimum** ($H\succ0$), **local maximum** ($H\prec0$), or **saddle point** (mixed signs). A
**global minimum** is the smallest value $L$ attains anywhere, not just locally; a non-global
local minimum is *spurious* — what early deep learning skeptics worried gradient descent would
get stuck in.

### Saddle points, not bad local minima, are the real obstacle

Two results replaced that worry. In high dimensions, a random critical point's Hessian behaves
like a random symmetric matrix, and the odds all $n$ eigenvalues share a sign shrink
exponentially in $n$ (Dauphin et al., 2014; Choromanska et al., 2015) — with millions of
parameters, an all-positive Hessian at a random critical point is astronomically unlikely, so
almost every critical point a large network meets is a saddle. And saddles are unstable: a
nearby point generically has some component along a negative-curvature direction that gradient
descent amplifies away from it — Lee et al. (2016) proved convergence to a strict saddle happens
with probability zero from a random start. Saddles still slow training (the gradient is tiny
nearby — part of why momentum, `04.04`, matters), but they don't trap you permanently.

### Overparameterization reshapes the landscape

As a network gets **wider** relative to what it needs to fit, bad local minima become rare and
the ones that remain sit close in value to the global minimum (proved for deep linear networks
by Kawaguchi, 2016; observed far more broadly) — more parameters means more directions are
available everywhere, so it's harder for *every* direction to curve upward at once. This is a
core reason modern networks are trained overparameterized rather than as small as possible.

### Symmetries you shouldn't mistake for "many solutions"

Permuting a hidden layer's units (relabeling neuron $i$ as $j$, swapping their weights) leaves
the loss unchanged — there are $n_{\text{hidden}}!$ identical copies of every minimum. ReLU
networks add a second symmetry: scaling one layer's incoming weights by $c>0$ and the next
layer's outgoing weights by $1/c$ also leaves the output unchanged, since
$\text{ReLU}(cz)=c\cdot\text{ReLU}(z)$ for $c>0$. Neither is "many solutions" — one solution,
different labels, verified in code below.

Non-convexity is real; nothing here makes the surface convex. What changes is the question: not
"can we escape every local minimum" but "how fast past saddles" (`04.04`), "how do we size a
step" (`04.06`), and "which near-equivalent minimum generalizes best" (`04.09`) — the thread
through the rest of Module `04`.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- needed to enable projection='3d'
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A single asymmetric double-well surface contains all three landmark types at once — a global
minimum, a shallower local minimum, and the saddle point separating them — which makes it the
perfect first landscape to look at before anything gets neural-network-sized.

In [ ]:
# f(x,y) = (x^2-1)^2 + 0.5*y^2 + 0.3*x -- an asymmetric "double well": two minima of different
# depth plus the saddle that separates them, all visible in one small surface.
def f(x, y):
    return (x**2 - 1) ** 2 + 0.5 * y**2 + 0.3 * x

def hess_f(x, y):
    return np.array([[12 * x**2 - 4, 0.0], [0.0, 1.0]])

# Critical points satisfy df/dx = 4x^3-4x+0.3 = 0 and df/dy = y = 0
crit_x_vals = np.sort(np.roots([4, 0, -4, 0.3]).real)
crit_points = [(x, 0.0) for x in crit_x_vals]

labels = []
for x, y in crit_points:
    eigvals = np.linalg.eigvalsh(hess_f(x, y))
    if np.all(eigvals > 0):
        labels.append("local minimum")
    elif np.all(eigvals < 0):
        labels.append("local maximum")
    else:
        labels.append("SADDLE POINT")

f_vals = [f(x, y) for x, y in crit_points]
global_idx = int(np.argmin(f_vals))
labels[global_idx] = "GLOBAL minimum"

x_grid = np.linspace(-2.0, 2.0, 150)
y_grid = np.linspace(-1.8, 1.8, 150)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

fig = plt.figure(figsize=(14, 5.5))
colors = {"GLOBAL minimum": "#55A868", "local minimum": "#DD8452", "SADDLE POINT": "#C44E52"}

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(X, Y, Z, cmap="viridis", alpha=0.75, linewidth=0, antialiased=True)
for (x, y), lab, fv in zip(crit_points, labels, f_vals):
    ax1.scatter([x], [y], [fv], color=colors[lab], s=70, depthshade=False)
ax1.set_xlabel("w1"); ax1.set_ylabel("w2"); ax1.set_zlabel("L(w1, w2)")
ax1.set_title("A loss surface with all three landmark types")
ax1.view_init(elev=28, azim=-60)

ax2 = fig.add_subplot(1, 2, 2)
ax2.contour(X, Y, Z, levels=25, cmap="viridis", alpha=0.7)
for (x, y), lab in zip(crit_points, labels):
    ax2.plot(x, y, "o", color=colors[lab], markersize=10, markeredgecolor="black")
    ax2.annotate(lab, (x, y), textcoords="offset points", xytext=(10, 8), fontsize=9)
ax2.set_xlabel("w1"); ax2.set_ylabel("w2")
ax2.set_title("Same surface from above, classified by Hessian eigenvalues")
plt.tight_layout()
plt.show()

for (x, y), lab, fv in zip(crit_points, labels, f_vals):
    eigvals = np.linalg.eigvalsh(hess_f(x, y))
    print(f"w1={x:+.4f}, w2={y:+.4f}  L={fv:+.4f}  Hessian eigs={np.round(eigvals, 3)}  -> {lab}")
print("\nThe local minimum (w1=+0.96) has higher loss than the global one (w1=-1.04) -- gradient")
print("descent started nearby converges there and never 'knows' a better point exists.")

## 🐍 Implementation from Scratch

We implement plain gradient descent from scratch, run it from many random starting points on
the double-well surface above, and map which basin each run falls into. Then we verify the
two landscape symmetries from the Theory section (permutation and ReLU scaling invariance) hold
exactly in code, and cross-check our converged points against `scipy.optimize.minimize`.

In [ ]:
from scipy.optimize import minimize

def gradient_descent(grad_fn, start, lr=0.02, steps=1000):
    # w <- w - lr*grad(w); which basin it lands in tells us which critical point 'caught' it
    point = np.array(start, dtype=float)
    for _ in range(steps):
        point = point - lr * grad_fn(*point)
    return point

def grad_f(x, y):
    return np.array([4 * x**3 - 4 * x + 0.3, y])

# 300 random starts -- a convex function sends every start to the SAME point; ours won't.
rng = np.random.default_rng(0)
n_runs = 300
starts = rng.uniform(-1.8, 1.8, size=(n_runs, 2))
endpoints = np.array([gradient_descent(grad_f, s) for s in starts])

global_min_x, saddle_x, local_min_x = crit_x_vals   # from the Visual Intuition cell, already sorted
in_global_basin = np.abs(endpoints[:, 0] - global_min_x) < 0.01
in_local_basin = np.abs(endpoints[:, 0] - local_min_x) < 0.01

print("Multi-start gradient descent on the double-well surface:")
print(f"  GLOBAL minimum basin: {in_global_basin.sum():3d}/{n_runs} ({100*in_global_basin.mean():.1f}%)")
print(f"  local minimum basin:  {in_local_basin.sum():3d}/{n_runs} ({100*in_local_basin.mean():.1f}%)")
print("Which basin you land in depends ENTIRELY on the start -- exactly what 'non-convex' means.")

# Cross-check against scipy's independent BFGS implementation
for x0, name in [(-1.5, "left well"), (1.5, "right well")]:
    ours = gradient_descent(grad_f, (x0, 0.3))
    scipy_result = minimize(lambda p: f(p[0], p[1]), x0=[x0, 0.3], jac=lambda p: grad_f(p[0], p[1]), method="BFGS")
    print(f"scipy cross-check ({name}): ours={np.round(ours, 4)}  scipy={np.round(scipy_result.x, 4)}  "
          f"match={np.allclose(ours, scipy_result.x, atol=1e-3)}")

# ---------- Verify the two landscape symmetries from the Theory section ----------
n_in, n_hidden = 3, 5
W1, b1 = rng.normal(size=(n_in, n_hidden)), rng.normal(size=n_hidden)
W2, b2 = rng.normal(size=n_hidden), rng.normal()
x_probe = rng.normal(size=(10, n_in))
tanh_net = lambda x, W1, b1, W2, b2: np.tanh(x @ W1 + b1) @ W2 + b2
relu_net = lambda x, W1, b1, W2, b2: np.maximum(0, x @ W1 + b1) @ W2 + b2

perm = rng.permutation(n_hidden)  # relabeling hidden units should change nothing computed
match_perm = np.allclose(tanh_net(x_probe, W1, b1, W2, b2), tanh_net(x_probe, W1[:, perm], b1[perm], W2[perm], b2))

c = 3.7  # relu(c*z)=c*relu(z) for c>0 -- scaling one layer up, the next down, is a FLAT direction
match_scale = np.allclose(relu_net(x_probe, W1, b1, W2, b2), relu_net(x_probe, W1 * c, b1 * c, W2 / c, b2))

print(f"Permuting hidden units -- output unchanged: {match_perm}")
print(f"Scaling ReLU layer by c=3.7, next by 1/c -- output unchanged: {match_scale}")
print("Raw minima counts overstate how many genuinely DIFFERENT solutions actually exist.")

## 🤖 Why This Matters for AI

Every real neural network — from a two-layer MLP to GPT-4 — trains by running gradient descent
(or a variant from `04.04`) on a landscape with exactly the structure above: full of saddles,
riddled with permutation/scaling symmetries, non-convex almost everywhere. Practitioners lean on
the overparameterization result deliberately: modern networks use *far* more parameters than
the minimum needed to fit the data, because wider networks empirically have fewer bad local
minima. Below, we train the same regression task at increasing hidden width and watch the
*worst-case* outcome across random initializations improve — evidence that "make it bigger" is
an optimization-landscape argument, not just a capacity argument.

In [ ]:
# Does WIDER really make bad local minima less of a problem? Train a 1-hidden-layer tanh net on
# a small regression task from many random inits, at increasing widths, tracking the WORST
# outcome per width -- one lucky seed reaching a good loss doesn't show the landscape improved.
N = 60
X_data = rng.uniform(-2, 2, size=(N, 2))
y_data = np.sin(2 * X_data[:, 0]) + np.cos(X_data[:, 1]) + rng.normal(0, 0.05, size=N)

def init_mlp(n_hidden, seed):
    r = np.random.default_rng(seed)
    return (r.normal(size=(2, n_hidden)) / np.sqrt(2), np.zeros(n_hidden),
            r.normal(size=n_hidden) / np.sqrt(n_hidden), 0.0)

def mlp_loss_and_grad(X, y, W1, b1, W2, b2):
    n = X.shape[0]
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    diff = A1 @ W2 + b2 - y
    loss = np.mean(diff ** 2)
    # backprop: squared error -> linear output -> tanh, the dense-layer formulas from 02.10 with
    # tanh's local gradient (1 - tanh(z)^2) in place of ReLU's
    d_yhat = 2 * diff / n
    dW2, db2 = A1.T @ d_yhat, d_yhat.sum()
    dZ1 = np.outer(d_yhat, W2) * (1 - A1 ** 2)
    dW1, db1 = X.T @ dZ1, dZ1.sum(axis=0)
    return loss, (dW1, db1, dW2, db2)

def train_mlp(n_hidden, seed, steps=1200, lr=0.05):
    W1, b1, W2, b2 = init_mlp(n_hidden, seed)
    for _ in range(steps):
        loss, (dW1, db1, dW2, db2) = mlp_loss_and_grad(X_data, y_data, W1, b1, W2, b2)
        W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2
    return loss

widths = [2, 4, 8, 16, 32, 64]
n_seeds = 15
final_losses = {w: [train_mlp(w, seed) for seed in range(n_seeds)] for w in widths}

print(f"{'width':>6} {'mean':>8} {'worst':>8} {'best':>8}")
for w in widths:
    losses = final_losses[w]
    print(f"{w:6d} {np.mean(losses):8.4f} {np.max(losses):8.4f} {np.min(losses):8.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(widths, [np.max(final_losses[w]) for w in widths], 'o-', color="#C44E52", label="worst-case (max over 15 seeds)")
ax.plot(widths, [np.mean(final_losses[w]) for w in widths], 'o-', color="#4C72B0", label="mean over 15 seeds")
ax.set_xscale("log", base=2)
ax.set_xlabel("hidden width (log scale)"); ax.set_ylabel("final training loss")
ax.set_title("Wider networks: the WORST unlucky init still ends\nup close to the best one")
ax.legend()
plt.show()

worst_at_2, worst_at_64 = np.max(final_losses[2]), np.max(final_losses[64])
print(f"worst-case loss at width=2: {worst_at_2:.4f}  ->  at width=64: {worst_at_64:.4f} "
      f"({worst_at_2/worst_at_64:.1f}x smaller)")
print("Overparameterization shrinks the WORST outcome across random inits -- the practical")
print("problem non-convexity actually poses -- even though no width makes the surface convex.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $h(x,y) = x^2 + xy + y^2 - 2x$, find the critical point by hand (solve $\nabla h=\mathbf{0}$),
   then compute the Hessian and classify the point using its eigenvalues. As a dimensional/sanity
   check, confirm $\det(H)$ equals the product of the eigenvalues and $\operatorname{tr}(H)$
   equals their sum, then verify everything with `np.linalg.eigvalsh`.

<details>
<summary>💡 Solution</summary>

$\nabla h = (2x+y-2,\ x+2y) = \mathbf{0}$: the second equation gives $x=-2y$; substituting into
the first, $2(-2y)+y-2=0 \Rightarrow -3y=2 \Rightarrow y=-2/3$, so $x=4/3$. The critical point is
$(4/3,-2/3)$, where $h=-4/3$. The Hessian is constant here:
$H=\begin{pmatrix}2&1\\1&2\end{pmatrix}$, with eigenvalues $1$ and $3$ — both positive, so this is
a local minimum, and since $h$ is a convex quadratic (positive-definite $H$ everywhere), it's the
global minimum too. Sanity check: $\det(H)=2\cdot2-1\cdot1=3=1\times3$ and
$\operatorname{tr}(H)=2+2=4=1+3$, both matching the eigenvalues exactly;
`np.linalg.eigvalsh([[2,1],[1,2]])` confirms $[1,3]$.

</details>

### 💭 UNDERSTAND
2. The saddle point in the Visual Intuition/Implementation cells sits at $w_1=$ `saddle_x`
   $\approx 0.0754$. Without running anything, predict what plain gradient descent does if
   initialized EXACTLY at this point, for any learning rate. Then predict whether initializing at
   `saddle_x + 1e-4` and at `saddle_x - 1e-4` end up in the *same* basin, using the Theory
   section's description of saddles as unstable. Verify all three cases in code.

<details>
<summary>💡 Solution</summary>

At exactly the saddle, $\nabla f(\text{saddle\_x}, 0)=\mathbf{0}$ by definition of a critical
point, so every gradient descent step is $w \leftarrow w - \eta\cdot\mathbf{0} = w$: the iterate
never moves, for *any* learning rate (verified: 2000 steps from exactly the saddle produce zero
drift). But this equilibrium is unstable, and the two tiny perturbations do NOT land in the same
basin: `saddle_x + 1e-4` has slightly larger $w_1$ and converges to the local minimum at
$w_1\approx0.960$; `saddle_x - 1e-4` has slightly smaller $w_1$ and converges to the global
minimum at $w_1\approx-1.036$ instead (both confirmed in code). The saddle's Hessian has a
negative eigenvalue along the $w_1$ direction, so any nonzero component along it gets *amplified*
by gradient descent rather than damped out — which basin you fall into is decided entirely by the
*sign* of an infinitesimal perturbation, exactly the instability the Theory section cites as why
saddles don't permanently trap gradient descent (a random initialization essentially never lands
exactly on the knife-edge).

</details>

### ✏️ DERIVE
3. Near a 1D local minimum $w^*$ with curvature $L=f''(w^*)>0$, gradient descent's error
   $e_t=w_t-w^*$ obeys, to first order, the linear recursion $e_{t+1}\approx(1-\eta L)e_t$. Derive
   the stability condition on $\eta$ this implies (the range of $\eta$ for which $e_t\to0$), then
   apply it to the local minimum at $w_1\approx0.960$ from the Visual Intuition cell (where
   $L=f''(w_1)=12w_1^2-4$) and verify numerically that gradient descent converges just below your
   derived threshold and fails to converge just above it.

<details>
<summary>💡 Solution outline</summary>

Linearize $f'$ around $w^*$: since $f'(w^*)=0$, a first-order Taylor expansion gives
$f'(w_t)\approx f''(w^*)(w_t-w^*)=L\,e_t$. Substitute into the update rule
$w_{t+1}=w_t-\eta f'(w_t)$ and subtract $w^*$ from both sides:
$e_{t+1}\approx e_t-\eta L e_t=(1-\eta L)e_t$. This linear recursion gives
$e_t\approx(1-\eta L)^t e_0$, which $\to0$ as $t\to\infty$ iff $|1-\eta L|<1$, i.e.
$-1<1-\eta L<1 \Rightarrow 0<\eta<2/L$ (using $L>0$). At $w_1\approx0.9601$,
$L=12(0.9601)^2-4\approx7.063$, giving threshold $\eta^\star=2/L\approx0.2832$. Verified: at
$\eta=0.9\,\eta^\star\approx0.2549$, an initial offset of $0.01$ from $w_1$ shrinks to exactly $0$
(floating-point precision) within 500 steps, and for a much tinier initial offset the ratio of
successive errors matches the predicted factor $1-\eta L=-0.8$ exactly at every step. At
$\eta=1.1\,\eta^\star\approx0.3115$, the same $0.01$ offset does NOT shrink to zero — it settles
into a persistent oscillation around $w_1$ (essentially identical statistics 500 steps in and 4000
steps in), confirming the derived threshold is exactly where convergent behavior stops.

</details>

### 🐍 IMPLEMENT
4. The Implementation cell verifies the ReLU scaling symmetry — $\text{ReLU}(cz)=c\cdot
   \text{ReLU}(z)$ for $c>0$ — using one fixed $c=3.7$ applied to the whole hidden layer at once.
   Extend it: for the notebook's own `W1, b1, W2` (5 hidden units), find, for *each* hidden unit
   $j$ independently, the rescaling $c_j$ that minimizes the L2 penalty
   $c_j^2\big(\|W_1[{:},j]\|^2+b_1[j]^2\big) + W_2[j]^2/c_j^2$ along that unit's own flat direction
   (i.e. rescale $W_1[{:},j], b_1[j] \to c_j\cdot(\cdot)$ and $W_2[j]\to W_2[j]/c_j$), checked
   against `scipy.optimize.minimize_scalar`.

<details>
<summary>✅ How to know you're right</summary>

Setting the derivative of $c^2a+b/c^2$ (with $a=\|W_1[{:},j]\|^2+b_1[j]^2$, $b=W_2[j]^2$) to zero
gives the closed form $c_j^\star=(b/a)^{1/4}$; this should match
`scipy.optimize.minimize_scalar`'s numerically-found minimizer for every unit to at least 4-5
decimal places. After rescaling every unit by its own $c_j^\star$, `relu_net`'s output on any
probe input must be **exactly** unchanged (to floating-point precision) from before rescaling —
that invariance is the entire point of a flat direction. Two things distinguish a correct
implementation from a subtly wrong one: (1) the bias $b_1[j]$ must be rescaled by the *same*
$c_j$ as $W_1[{:},j]$, since both feed into the same pre-activation $z=xW_1+b_1$ that ReLU
actually sees — scaling $W_1$ alone while leaving $b_1$ untouched breaks the invariance
(check this explicitly: the output changes if you do). (2) the total penalty
$\sum_j\big(\|W_1[{:},j]\|^2+b_1[j]^2+W_2[j]^2\big)$ should measurably *decrease* after balancing
every unit (by several tens of percent, for a typical random initialization), since $c=1$ is
essentially never a unit's own optimal balance point.

</details>

### 🤖 APPLY
5. A 2D quadratic bowl $f(x,y)=\tfrac12(L_xx^2+L_yy^2)$ with $L_x=1, L_y=50$ (a stand-in for a
   badly-conditioned loss, condition number 50) has two *independent* per-axis stability
   thresholds from your DERIVE exercise: $\eta<2/L_x=2$ and $\eta<2/L_y=0.04$. Using a single
   shared learning rate safe for *both* directions at once (i.e. respecting the tighter
   constraint — try $\eta=0.9\times2/L_y=0.036$), count how many steps each axis needs to shrink
   an initial offset of $1$ below $10^{-3}$, using the per-axis contraction factor $|1-\eta L|$
   from your derivation.

<details>
<summary>✅ What you should observe</summary>

The $y$-axis (high curvature, $L_y=50$) converges quickly — about 30 steps — while the $x$-axis
(low curvature, $L_x=1$) needs roughly 6 times as many steps (around 190) to reach the same
tolerance, purely because the single shared $\eta$ is forced to be tiny by $y$'s stability
requirement even though $x$ on its own could tolerate a step 50 times larger. If you compute the
$\eta$ that *would* make $x$ converge as fast as $y$ does at the shared rate (roughly $1.8$), it
sits far above $y$'s own threshold of $0.04$; plugging it into $y$'s contraction factor gives
$|1-\eta L_y|\approx89$, i.e. genuine divergence, not just slower convergence. This is the
concrete mechanism behind "ill-conditioned loss surfaces slow down gradient descent": one
direction's stability requirement can force a learning rate that leaves every *other*,
better-conditioned direction converging far slower than it's capable of on its own — exactly what
motivates the per-parameter adaptive step sizes `04.04` introduces next.

</details>

### 🚀 CHALLENGE
6. The Theory section notes that permuting a hidden layer's units leaves the loss unchanged, so
   there are at least $n_{\text{hidden}}!$ identical copies of every minimum. First, confirm this
   holds for ALL $5!=120$ permutations of the notebook's own 5-unit `tanh_net` (not just the
   Implementation cell's one random check). Then investigate a consequence: train two *separate*
   tanh networks (same architecture, different random seeds) on the width-experiment's regression
   task, and naively average their weights directly. Compare that to averaging them AFTER first
   solving an assignment problem (e.g. `scipy.optimize.linear_sum_assignment`) that matches each
   hidden unit in network A to its most-similar counterpart in network B, then permuting B's units
   to align with A before averaging. Does matching first help, and does the averaged network ever
   match or beat either original network alone?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms all 120 permutations leave `tanh_net`'s output exactly unchanged (not
just the single permutation the Implementation cell happens to test), correctly reading
$n_{\text{hidden}}!$ as a *lower bound* on the number of loss-preserving copies of any one
minimum. On the averaging investigation: naive averaging (no permutation matching) typically
produces a network dramatically worse than either original — in one run, two networks with
training losses around $0.03$ and $0.02$ averaged naively to a loss over $3.5$ — because two
independently trained networks almost certainly settle into different, unrelated orderings of
their hidden units, so averaging "unit 3 of A" with "unit 3 of B" blends two units that learned
unrelated features, not a sensible compromise. Matching permutations first (via the assignment
problem) recovers most of this loss — often landing more than an order of magnitude better than
the naive average — but a strong answer notices it typically still doesn't match or beat *either*
original network's own loss. This connects to `04.09`'s later discussion of mode connectivity:
permutation symmetry is only *one* of the landscape's structures, and even after accounting for
it, two independently-trained solutions need not be connected by a low-loss straight-line path (a
full linear interpolation, of which averaging is the midpoint) — so nothing guarantees the matched
midpoint itself sits in a good place. A strong answer treats this as motivation for why real
model-merging techniques (federated-learning aggregation, "model soups," Git Re-Basin) need to
solve the permutation-matching problem explicitly rather than average raw checkpoints.

</details>

---

## 📚 Further Reading
- Dauphin et al., ["Identifying and Attacking the Saddle Point Problem in High-Dimensional Non-Convex Optimization"](https://arxiv.org/abs/1406.2572)
- Choromanska et al., ["The Loss Surfaces of Multilayer Networks"](https://arxiv.org/abs/1412.0233)
- Lee et al., ["Gradient Descent Only Converges to Minimizers"](https://arxiv.org/abs/1602.04915) (COLT 2016)
- Kawaguchi, ["Deep Learning without Poor Local Minima"](https://arxiv.org/abs/1605.07110)

---

*Next notebook: [Gradient Descent and Its Variants](02_gradient_descent_variants.ipynb)*